# Package StrawGolem Mod in Google Colab

This notebook guides you through packaging the StrawGolem Minecraft mod into JAR files using Google Colab. It covers setting up the environment, cloning the repository, building with Gradle, and downloading the resulting JARs.

## 1. Setup Google Colab Environment

This section prepares your Google Colab environment. We'll mount Google Drive to potentially save artifacts and ensure we have a working directory. However, for this specific task of building a JAR from a Git repository, direct Drive mounting for the build process itself is optional unless you want to store the final JARs there persistently or have the source code already in Drive.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive (optional, but good practice for saving outputs)
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
    # You can create a working directory in your Drive if you wish
    # working_dir = "/content/drive/My Drive/Colab Notebooks/StrawGolemBuild"
    # if not os.path.exists(working_dir):
    #     os.makedirs(working_dir)
    # os.chdir(working_dir)
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    print("Proceeding without Google Drive mount. Files will be stored in the temporary Colab environment.")

# Set a local working directory in Colab's environment
local_working_dir = "/content/strawgolem_build"
if not os.path.exists(local_working_dir):
    os.makedirs(local_working_dir)
os.chdir(local_working_dir)
print(f"Current working directory: {os.getcwd()}")

## 2. Clone the Repository

Next, we'll clone the StrawGolem mod repository from GitHub (or your specified Git provider) into the Colab environment.

**Important:** Replace `YOUR_REPOSITORY_URL` with the actual HTTPS URL of the StrawGolem Git repository.

In [ ]:
# Ensure git is available (usually is on Colab)
!apt-get update -qq > /dev/null
!apt-get install -y git -qq > /dev/null

# !!! Replace YOUR_REPOSITORY_URL with the actual repository URL !!!
repository_url = "YOUR_REPOSITORY_URL"  # e.g., "https://github.com/user/strawgolem.git"
project_dir_name = "strawgolem"

if repository_url == "YOUR_REPOSITORY_URL" or not repository_url:
    print("ERROR: Please replace 'YOUR_REPOSITORY_URL' with the actual Git repository URL.")
else:
    if os.path.exists(project_dir_name):
        print(f"Directory '{project_dir_name}' already exists. Removing it to ensure a fresh clone...")
        !rm -rf {project_dir_name}
    print(f"Cloning repository: {repository_url}")
    !git clone {repository_url} {project_dir_name}
    
    if os.path.exists(project_dir_name):
        print(f"Repository cloned successfully into '{project_dir_name}'.")
        os.chdir(project_dir_name) # Change into the project directory
        print(f"Current directory: {os.getcwd()}")
        !ls -la
    else:
        print("Error: Cloning failed or directory not found post-clone.")

## 3. Install Java and Grant Gradle Permissions

Minecraft modding, especially for older versions or specific setups, often requires a particular Java Development Kit (JDK) version. We'll install OpenJDK 17, which is commonly used. Then, we need to give the Gradle wrapper (`gradlew`) execute permissions.

*Note: The `gradle.properties` file in the repository specifies `minecraft_version=1.21.1`. Ensure the JDK version is compatible.*

In [ ]:
# Install OpenJDK 17
print("Installing OpenJDK 17...")
!apt-get update -qq > /dev/null
!apt-get install -y openjdk-17-jdk -qq > /dev/null
print("OpenJDK 17 installation attempt complete.")

# Verify Java installation
!java -version

# Set JAVA_HOME environment variable (important for Gradle)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print(f"JAVA_HOME set to: {os.environ['JAVA_HOME']}")

# Grant execute permissions to gradlew
# This assumes you are in the root of the cloned repository
if os.path.exists("gradlew"):
    !chmod +x gradlew
    print("Execute permissions granted to ./gradlew")
else:
    print("Error: gradlew script not found. Make sure you are in the project's root directory.")
    print(f"Current directory: {os.getcwd()}")

## 4. Build the JAR Files

With the environment set up, we can now use the Gradle wrapper to build the project. This will compile the code and package it into JAR files for both Fabric and Forge mod loaders, as defined in the project's build scripts.

This step can take several minutes as Gradle downloads dependencies and compiles the code.

In [ ]:
# Run the Gradle build command
# This command should build both Fabric and Forge versions if the project is configured to do so.
if os.path.exists("./gradlew"):
    print("Starting Gradle build... This may take a significant amount of time.")
    # The `build` task usually creates the JARs.
    # Adding --no-daemon can sometimes help in CI environments like Colab.
    !./gradlew build --no-daemon
    print("Gradle build process finished.")
else:
    print("Error: gradlew script not found. Cannot start build.")

## 5. Locate and Download the JAR Files

After a successful build, the JAR files are typically found in the `build/libs` subdirectory of each subproject (e.g., `fabric/build/libs` and `forge/build/libs`).

The `gradle.properties` file indicates `version=2.2.0` and `mod_id=strawgolem`.
So, the expected JAR names would be something like `strawgolem-fabric-2.2.0.jar` and `strawgolem-forge-2.2.0.jar`.

Let's list the potential output directories and then provide a way to download them.

In [ ]:
from google.colab import files

# Define expected JAR paths and names (based on gradle.properties and common project structure)
version = "2.2.0" # From gradle.properties
mod_id = "strawgolem" # From gradle.properties

fabric_jar_name = f"{mod_id}-fabric-{version}.jar"
forge_jar_name = f"{mod_id}-forge-{version}.jar"

fabric_jar_path = f"fabric/build/libs/{fabric_jar_name}"
forge_jar_path = f"forge/build/libs/{forge_jar_name}"

print("--- Checking for Fabric JAR ---")
if os.path.exists(fabric_jar_path):
    print(f"Found Fabric JAR: {fabric_jar_path}")
    !ls -lh {fabric_jar_path}
    print(f"\nAttempting to download {fabric_jar_name}...")
    files.download(fabric_jar_path)
else:
    print(f"Fabric JAR NOT FOUND at: {fabric_jar_path}")
    print("Please check the build logs and the fabric/build/libs/ directory content:")
    if os.path.exists("fabric/build/libs/"):
        !ls -l fabric/build/libs/
    else:
        print("fabric/build/libs/ directory does not exist.")

print("\n--- Checking for Forge JAR ---")
if os.path.exists(forge_jar_path):
    print(f"Found Forge JAR: {forge_jar_path}")
    !ls -lh {forge_jar_path}
    print(f"\nAttempting to download {forge_jar_name}...")
    files.download(forge_jar_path)
else:
    print(f"Forge JAR NOT FOUND at: {forge_jar_path}")
    print("Please check the build logs and the forge/build/libs/ directory content:")
    if os.path.exists("forge/build/libs/"):
        !ls -l forge/build/libs/
    else:
        print("forge/build/libs/ directory does not exist.")

print("\n--- Build and Download Process Complete ---")
print("If downloads did not start, check the output above for errors or missing files.")